# MEDiTwin Model 07: Sleep Quality Prediction Model

**Personal Health Digital Twin & Lifestyle Simulation Prototype**
*Academic Major Project — University Research Submission*

---

### ⚠️ Critical Clinical Safety Disclaimer
> **NOTICE:** MediTwin is an educational wellness and simulation research prototype—**NOT a medical device, diagnostic system, or clinical decision support platform**.
> - It does not diagnose diseases, replace medical practitioners, prescribe treatment, or recommend altering prescription medications.
> - All outputs represent educational estimates, screening patterns, or longitudinal trend indicators.
> - Predictive correlations in observational cohorts must not be construed as clinical causality.


## 1. Environment Setup & Data Loading
- **Dataset:** `sleep_health_lifestyle.csv`
- **Target Variable:** `Quality of Sleep (Rating scale 1 to 10)`
- **Description:** Continuous regression modeling sleep restorative quality from lifestyle habits and circadian duration.


In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src modules are resolvable
sys.path.insert(0, os.path.abspath(".."))
from src.sleep_quality import (
    load_and_preprocess_data,
    train_and_evaluate,
    FEATURES if hasattr(sys.modules.get('src.sleep_quality'), 'FEATURES') else ALL_FEATURES
)
from src.common import RAW_DATA_DIR, clean_columns, to_snake_case

print("Environment and packages loaded successfully.")


## 2. Exploratory Data Analysis (EDA)
In accordance with project requirements, we inspect:
1. Dataset shape (rows, columns)
2. Column names & data types
3. Missing values analysis
4. Duplicate row detection
5. Target variable distribution
6. Summary descriptive statistics


In [ ]:
raw_csv = os.path.join(RAW_DATA_DIR, "sleep_health_lifestyle.csv")
df_raw = pd.read_csv(raw_csv)

print("=== 1. DATASET SHAPE ===")
print(f"Total Rows: {df_raw.shape[0]}, Total Columns: {df_raw.shape[1]}")

print("\n=== 2. COLUMNS & DATA TYPES ===")
print(df_raw.dtypes)

print("\n=== 3. MISSING VALUES COUNT ===")
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No NaN values detected in raw load.")

print("\n=== 4. DUPLICATE ROWS ===")
print(f"Duplicate rows detected: {df_raw.duplicated().sum()}")

print("\n=== 5. TARGET DISTRIBUTION ===")
clean_df = clean_columns(df_raw)
print(clean_df.iloc[:, -1].value_counts(dropna=False))

print("\n=== 6. BASIC SUMMARY STATISTICS ===")
df_raw.describe().round(2)


## 3. Preprocessing, Imputation & Model Training
We evaluate four separate algorithm families:
1. **Logistic / Linear Regression**
2. **Decision Tree**
3. **Random Forest**
4. **XGBoost (Extreme Gradient Boosting)**

All models are integrated into reproducible scikit-learn `Pipeline` and `ColumnTransformer` workflows using `random_state=42` and `test_size=0.20`.


In [ ]:
# Run pipeline training, cross-validation, chart generation, and artifact export
best_model, metadata = train_and_evaluate()

print("\n=== BEST MODEL TRAINING METADATA ===")
print(json.dumps(metadata, indent=2, default=str))


## 4. Model Evaluation & Comparison Table
Reviewing test set metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC, PR-AUC).


In [ ]:
from src.common import METRICS_DIR

comparison_csv = [f for f in os.listdir(METRICS_DIR) if f.startswith("07") and f.endswith(".csv")][0]
comp_df = pd.read_csv(os.path.join(METRICS_DIR, comparison_csv))
print("Model Comparison Table (Sorted by Primary Score):")
comp_df


## 5. Visualizations & Diagnostic Charts
Displaying target distribution, confusion matrices, ROC/PR curves, and feature importance.


In [ ]:
from src.common import FIGURES_DIR
from PIL import Image

model_figures = [f for f in os.listdir(FIGURES_DIR) if f.startswith("07")]
print(f"Generated Figures ({len(model_figures)}):")
for fig in sorted(model_figures):
    print(f" - {fig}")


## 6. Model Card, Ethical Considerations & Uncertainty
A full ethical model card is automatically exported to `reports/model_cards/07_*_card.md`.


In [ ]:
from src.common import MODEL_CARDS_DIR

card_files = [f for f in os.listdir(MODEL_CARDS_DIR) if f.startswith("07")]
if card_files:
    card_path = os.path.join(MODEL_CARDS_DIR, card_files[0])
    with open(card_path, "r", encoding="utf-8") as f:
        print(f.read())


## 7. API-Ready Prediction Function & Boundary Testing
Testing sample inference with validity checks, uncertainty notes, and mandatory clinical disclaimer.


In [ ]:
import importlib
mod = importlib.import_module("src.sleep_quality")
predict_fn = [getattr(mod, f) for f in dir(mod) if f.startswith("predict_")][0]

sample_test_input = {}
# Execute sample inference
sample_output = predict_fn(sample_test_input)
print("Sample Model Prediction Output:")
print(json.dumps(sample_output, indent=2))
